<a href="https://colab.research.google.com/github/saimouzhang-research/Reuse-of-new-named-entity-task/blob/main/Entire%20script_Mar%2021.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# =========================
# 1. Download / load data
# =========================
!wget https://cs.stanford.edu/~minalee/zip/chi2022-coauthor-v1.0.zip
!unzip -q chi2022-coauthor-v1.0.zip
!rm chi2022-coauthor-v1.0.zip


--2026-03-21 15:16:33--  https://cs.stanford.edu/~minalee/zip/chi2022-coauthor-v1.0.zip
Resolving cs.stanford.edu (cs.stanford.edu)... 171.64.64.64
Connecting to cs.stanford.edu (cs.stanford.edu)|171.64.64.64|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 49956179 (48M) [application/zip]
Saving to: ‘chi2022-coauthor-v1.0.zip’

chi2022-coauthor-v1 100%[===================>]  47.64M  28.4MB/s    in 1.7s    

2026-03-21 15:16:34 (28.4 MB/s) - ‘chi2022-coauthor-v1.0.zip’ saved [49956179/49956179]



In [ ]:

import os
import json
import pandas as pd
import nltk
from nltk.tokenize import sent_tokenize

nltk.download('punkt')
nltk.download('punkt_tab')

# =========================
# 2. Read session paths
# =========================
def find_writing_sessions(dataset_dir):
    return [
        os.path.join(dataset_dir, path)
        for path in os.listdir(dataset_dir)
        if path.endswith('jsonl')
    ]

def read_writing_session(path):
    events = []
    with open(path, 'r') as f:
        for event in f:
            events.append(json.loads(event))
    return events

dataset_dir = './coauthor-v1.0'
paths = find_writing_sessions(dataset_dir)

# =========================
# 3. Load metadata
# =========================
def sess2auth(zip_list):
    out_dict = {}
    for author, session in zip_list:
        if session not in out_dict:
            out_dict[session.strip()] = author.strip()
    return out_dict

df_a = pd.read_csv('/content/CoAuthor - Metadata & Survey - Metadata (argumentative).csv')
df_c = pd.read_csv('/content/CoAuthor - Metadata & Survey - Metadata (creative).csv')

session_id_col = 'session_id'
worker_id_col = 'worker_id'

df_a_list = list(zip(df_a[worker_id_col], df_a[session_id_col]))
df_c_list = list(zip(df_c[worker_id_col], df_c[session_id_col]))

sess_auth_dict = sess2auth(df_a_list + df_c_list)

# =========================
# 4. Reconstruct text + mask
# =========================
def apply_ops(doc, mask, ops, source):
    original_doc = doc
    original_mask = mask

    new_doc = ''
    new_mask = ''

    for op in ops:
        if 'retain' in op:
            num_char = op['retain']
            retain_doc = original_doc[:num_char]
            retain_mask = original_mask[:num_char]

            original_doc = original_doc[num_char:]
            original_mask = original_mask[num_char:]

            new_doc += retain_doc
            new_mask += retain_mask

        elif 'insert' in op:
            insert_doc = op['insert']

            if isinstance(insert_doc, dict):
                # skip non-text insertions
                continue

            insert_mask = 'A' * len(insert_doc) if source == 'api' else 'U' * len(insert_doc)

            new_doc += insert_doc
            new_mask += insert_mask

        elif 'delete' in op:
            num_char = op['delete']

            if original_doc:
                original_doc = original_doc[num_char:]
                original_mask = original_mask[num_char:]
            else:
                new_doc = new_doc[:-num_char]
                new_mask = new_mask[:-num_char]

    final_doc = new_doc + original_doc
    final_mask = new_mask + original_mask
    return final_doc, final_mask

def get_text_and_mask(events, event_id, remove_prompt=False):
    prompt = events[0]['currentDoc'].strip()

    text = prompt
    mask = 'P' * len(prompt)

    for event in events[:event_id]:
        if 'textDelta' not in event or 'ops' not in event['textDelta']:
            continue

        ops = event['textDelta']['ops']
        source = event.get('eventSource', 'user')
        text, mask = apply_ops(text, mask, ops, source)

    if remove_prompt and 'P' in mask:
        end_index = mask.rindex('P')
        text = text[end_index + 1:]
        mask = mask[end_index + 1:]

    return text, mask

# =========================
# 5. Prompt ID mapping
# =========================
prompt_shapeshifter = {''.join('A woman has'.split()): 'shapeshifter'}
prompt_reincarnation = {''.join('When you die,'.split()): 'reincarnation'}
prompt_mana = {''.join('Humans once wielded'.split()): 'mana'}
prompt_obama = {''.join("You're Barack Obama.".split()): 'obama'}
prompt_pig = {''.join('Once upon a'.split()): 'pig'}
prompt_mattdamon = {''.join('An alien has'.split()): 'mattdamon'}
prompt_sideeffect = {''.join("When you're 28,".split()): 'sideeffect'}
prompt_bee = {''.join("Your entire life,".split()): 'bee'}
prompt_dad = {''.join("All of the".split()): 'dad'}
prompt_isolation = {''.join("Following World War".split()): 'isolation'}
prompt_screen = {''.join('How Worried Should'.split()): 'screen'}
prompt_dating = {''.join('How Do You'.split()): 'dating'}
prompt_pads = {''.join('Should Schools Provide'.split()): 'pads'}
prompt_school = {''.join('What Are the'.split()): 'school'}
prompt_stereotype = {''.join('What Stereotypical Characters'.split()): 'stereotype'}
prompt_audiobook = {''.join('Is Listening to'.split()): 'audiobook'}
prompt_athletes = {''.join('Should College Athletes'.split()): 'athletes'}
prompt_extremesports = {''.join('Is It Selfish'.split()): 'extremesports'}
prompt_animal = {''.join('Is It Wrong'.split()): 'animal'}
prompt_news = {''.join("Are We Being".split()): 'news'}

prompt_dict = (
    prompt_isolation
    | prompt_dad
    | prompt_shapeshifter
    | prompt_reincarnation
    | prompt_mana
    | prompt_obama
    | prompt_pig
    | prompt_mattdamon
    | prompt_sideeffect
    | prompt_bee
    | prompt_screen
    | prompt_dating
    | prompt_pads
    | prompt_school
    | prompt_stereotype
    | prompt_audiobook
    | prompt_athletes
    | prompt_extremesports
    | prompt_animal
    | prompt_news
)

def get_prompt_id_from_text(text, prompt_dict=prompt_dict):
    prompt_first = ''.join(text.split()[:3])
    return prompt_dict.get(prompt_first, 'misc')

# =========================
# 6. Sentence-level masks
# =========================
def classify_sentences_with_mask_flat(text, mask):
    sentence_rows = []
    search_start = 0

    for sentence_id, sentence in enumerate(sent_tokenize(text.strip())):
        sentence = sentence.strip()
        if not sentence:
            continue

        start_idx = text.find(sentence, search_start)
        if start_idx == -1:
            start_idx = text.find(sentence)
        if start_idx == -1:
            print(f'Could not find sentence in text: {sentence}')
            continue

        end_idx = start_idx + len(sentence)
        sentence_mask = mask[start_idx:end_idx]

        chars = set(sentence_mask)
        if chars == {'A'}:
            sentence_author = 'api'
        elif chars == {'U'}:
            sentence_author = 'user'
        else:
            sentence_author = 'user_and_api'

        sentence_rows.append({
            'sentence_id': sentence_id,
            'sentence_text': sentence,
            'sentence_mask': sentence_mask,
            'sentence_author': sentence_author
        })

        search_start = end_idx

    return sentence_rows

# =========================
# 7. Build final dataframe
# =========================
rows = []

for path in paths:
    session_id = path.split('/')[2].split('.')[0].strip()
    author_id = sess_auth_dict.get(session_id, 'missing_info')

    events = read_writing_session(path)
    text, mask = get_text_and_mask(events, len(events), remove_prompt=False)
    prompt_id = get_prompt_id_from_text(text)
    sentence_with_masks = classify_sentences_with_mask_flat(text, mask)

    rows.append({
        'session_id': session_id,
        'author_id': author_id,
        'prompt_id': prompt_id,
        'text': text,
        'mask': mask,
        'sentence_with_masks': json.dumps(sentence_with_masks, ensure_ascii=False)
    })

df_out = pd.DataFrame(rows)
df_out.to_csv('CoAuthor_session_text_masks.csv', index=False)

print(df_out.head())
print(df_out.columns)
print(df_out.shape)